# SpectraBreast 3D Reconstruction Workshop

This notebook contains organically written, possibly non-functional debugging code. 

In [ ]:
import torch
import torchvision.io as io
from pathlib import Path
from helpers import *
import numpy as np
from rich import print
# Assume CUDA is available based on user rules
device = torch.device("cuda")
print("Using device:", f"[green]{device}[/green]" if device.type == "cuda" else f"[red]{device}[/red]")

from helpers import xyzeuler_to_hmat
rgb_image_dir = Path("rgb_images/")
camera_pose_dir = Path("camera_poses/")
image_files = sorted(rgb_image_dir.glob("image_*.png"))
pose_files = sorted(camera_pose_dir.glob("pose_*.txt"))

# Load images as [C, H, W] tensors to GPU
images = [io.read_image(str(p)).to(device) for p in image_files]

# Load poses as [6] tensors to GPU
poses = [
    torch.tensor(
        [float(x) for x in p.read_text().strip().split()],
        dtype=torch.float32,
        device=device,
    )
    for p in pose_files
]

# Batch everything if non-empty
if images and poses:
    images_tensor = torch.stack(images)
    poses_tensor = torch.stack(poses)
    print(f"Loaded {len(images)} pairs.")
    print(f"Batched images shape: {images_tensor.shape}")
    print(f"Batched poses6 shape: {poses_tensor.shape}")
translation_scale = 1
poses_tensor4 = xyzeuler_to_hmat(
    poses_tensor, convention="ROLLPITCHYAW", translation_scale=translation_scale
)
print(f"Batched poses shape: {poses_tensor4.shape}")

# Load intrinsics and cam2ee, expand as batch
intrinsics = np.load("camera_parameters/intrinsics.npy")
intrinsics = (
    torch.tensor(intrinsics)
    .cuda()
    .unsqueeze(0)
    .repeat(len(poses_tensor), 1, 1)
)
cam2ee = np.load("camera_parameters/camera2ee.npy")

# ---- Fix: cam2ee must be (4,4) but was (3,4) ----
# We'll check and expand if needed. 
if cam2ee.shape == (3, 4):
    # Make last row [0, 0, 0, 1], so (4, 4)
    cam2ee_fixed = np.eye(4, dtype=np.float32)
    cam2ee_fixed[:3, :4] = cam2ee
    cam2ee = cam2ee_fixed

cam2ee = torch.tensor(cam2ee).cuda().unsqueeze(0).repeat(len(poses_tensor), 1, 1)
# Now cam2ee is [N, 4, 4] and poses_tensor4 is [N, 4, 4]
poses_tensor_prem = torch.matmul(poses_tensor4, cam2ee)
print(f"Batched intrinsic shape: {intrinsics.shape}")


cuda


In [26]:
import torch
from depth_anything_3.api import DepthAnything3

# Load model from Hugging Face Hub
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DepthAnything3.from_pretrained("depth-anything/da3-giant")
model = model.to(device=device)

# Run inference on images
prediction = model.inference(
    [str(path) for path in image_files],
    export_dir="output",
    process_res=640,
    # extrinsics=torch.linalg.inv(poses_tensor_prem).cpu().numpy(),
    intrinsics=intrinsics.cpu().numpy(),
    align_to_input_ext_scale=True,
    export_format="glb",
    show_cameras=True,
)

import plotly.graph_objects as go
import trimesh
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
import numpy as np
from pathlib import Path

# =====================================================================
# 1. Load data from the exported NPZ file
# =====================================================================
npz_path = Path("/home/arota/spectra/output/exports/npz/results.npz")
if npz_path.exists():
    data = np.load(str(npz_path))
data = np.load(str(npz_path))


[INFO ] using SwiGLU layer as FFN
[INFO ] Processed Images Done taking 0.0966792106628418 seconds. Shape:  torch.Size([20, 3, 476, 644])
[INFO ] Selecting reference view using strategy: saddle_balanced
[INFO ] Model Forward Pass Done. Time: 2.5755972862243652 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0054225921630859375 seconds
[INFO ] conf_thresh_percentile: 40.0
[INFO ] num max points: 1000000
[INFO ] Exporting to GLB with num_max_points: 1000000
[INFO ] Export Results Done. Time: 1.5343821048736572 seconds


In [ ]:
import trimesh
import numpy as np


def load_glb_mesh(file_path: str, export_path: str = None) -> trimesh.Trimesh:
    """
    Loads a .glb file. If it contains a point cloud without faces, it will handle it correctly.
    Optionally exports the resulting geometry. For point clouds, '.ply' is highly recommended over '.obj'.

    Args:
        file_path (str): The path to the .glb file.
        export_path (str, optional): The path where the mesh/points should be saved. Defaults to None.

    Returns:
        trimesh.Trimesh or trimesh.points.PointCloud: The extracted 3D geometry object.
    """
    # Load the scene
    loaded_data = trimesh.load(file_path)

    # Check if the loaded data is a Scene and extract its geometries
    if isinstance(loaded_data, trimesh.Scene):
        # Dump concatenates everything into a single mesh or point cloud
        geometry = loaded_data.dump(concatenate=True)
    else:
        geometry = loaded_data

    # Export logic
    if export_path is not None:
        # Check if geometry has faces. If len(faces) == 0, it's a Point Cloud
        has_faces = hasattr(geometry, "faces") and len(geometry.faces) > 0

        if not has_faces and export_path.lower().endswith(".obj"):
            print(
                "Warning: Exporting a Point Cloud to .obj. Writing points manually as 'v x y z'..."
            )
            # Trimesh export to OBJ requires faces. For point clouds, we must write it manually.
            with open(export_path, "w") as f:
                for v in geometry.vertices:
                    f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n")
            print(f"Point Cloud vertices successfully saved to: {export_path}")

        else:
            # Standard export (e.g., if it's a real mesh, or exporting to .ply)
            geometry.export(export_path)
            print(f"Geometry successfully saved to: {export_path}")

    return geometry


# Example usage to load and save:
geometry = load_glb_mesh(
    file_path="/home/arota/spectra/output/scene.glb",
    export_path="/home/arota/spectra/output/scene.obj",  # Or consider using .ply
)

Point Cloud vertices successfully saved to: /home/arota/spectra/output/scene.obj


In [22]:
# Extract the relevant arrays
# depth: (N, H, W)
# image: (N, H, W, 3)  (assuming RGB is stored)
# exts: (N, 4, 4) or (N, 3, 4) world-to-camera matrices
# ixts: (N, 3, 3) camera intrinsics
# conf: (N, H, W) confidence maps (optional but usually present)
depths = data["depth"]
images = data["image"]
exts = data["extrinsics"]
ixts = data["intrinsics"]

# Ensure extrinsics are 4x4
if exts.shape[1:] == (3, 4):
    bottom_row = np.array([[[0, 0, 0, 1]]])
    bottom_row = np.repeat(bottom_row, exts.shape[0], axis=0)
    exts = np.concatenate([exts, bottom_row], axis=1)

# Invert world-to-camera to get camera-to-world (poses)
glb_camera_poses = np.linalg.inv(exts)

# =====================================================================
# 2. Unproject dense depth maps into a 3D Point Cloud
# =====================================================================
all_points = []
all_colors = []

N, H, W = depths.shape

# Subsampling factor for depth maps to avoid out-of-memory/browser crashes
# Adjust this based on your needs (e.g., skip=4 means taking 1 out of every 16 pixels)
skip = 8

# Create meshgrid for pixel coordinates
u, v = np.meshgrid(np.arange(0, W, skip), np.arange(0, H, skip))
u = u.flatten()
v = v.flatten()
ones = np.ones_like(u)

for i in range(N):
    depth = depths[i, ::skip, ::skip].flatten()

    # Optional: Filter out low confidence points if confidence map exists
    if "conf" in data:
        conf = data["conf"][i, ::skip, ::skip].flatten()
        # Keep points with confidence > median
        mask = conf > np.percentile(conf, 40)
    else:
        # Keep points with positive depth
        mask = depth > 0.1

    d = depth[mask]
    pts_u = u[mask]
    pts_v = v[mask]

    fx, fy = ixts[i, 0, 0], ixts[i, 1, 1]
    cx, cy = ixts[i, 0, 2], ixts[i, 1, 2]

    # Unproject to camera coordinates
    x_cam = (pts_u - cx) * d / fx
    y_cam = (pts_v - cy) * d / fy
    z_cam = d

    pts_cam = np.stack([x_cam, y_cam, z_cam, np.ones_like(x_cam)], axis=1)  # (M, 4)

    # Transform to world coordinates using camera-to-world pose
    pose = glb_camera_poses[i]
    pts_world = (pose @ pts_cam.T).T[:, :3]  # (M, 3)

    all_points.append(pts_world)

    # Get colors from the image
    img = images[i, ::skip, ::skip].reshape(-1, 3)[mask]
    all_colors.append(img)

# Combine points from all views
vertices = np.concatenate(all_points, axis=0)
colors = np.concatenate(all_colors, axis=0)

# Global subsampling if still too large for Plotly
MAX_POINTS = 100_000
if len(vertices) > MAX_POINTS:
    indices = np.random.choice(len(vertices), MAX_POINTS, replace=False)
    vertices = vertices[indices]
    colors = colors[indices]

marker_colors = [f"rgb({c[0]},{c[1]},{c[2]})" for c in colors]

# =====================================================================
# 3. Extract Your Original Camera Poses (poses_tensor_prem)
# =====================================================================
poses_np = poses_tensor_prem.detach().cpu().numpy()

translations = poses_np[:, :3, 3]
rotations = poses_np[:, :3, :3]

x_dirs = rotations[:, :, 0]
y_dirs = rotations[:, :, 1]
z_dirs = rotations[:, :, 2]

# =====================================================================
# 4. Create the combined Plotly Figure
# =====================================================================
fig = go.Figure()

# Add Point Cloud
fig.add_trace(
    go.Scatter3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        mode="markers",
        marker=dict(size=2, color=marker_colors, opacity=0.8),
        name="NPZ Point Cloud",
    )
)

# Add Original Camera Centers
fig.add_trace(
    go.Scatter3d(
        x=translations[:, 0],
        y=translations[:, 1],
        z=translations[:, 2],
        mode="markers",
        marker=dict(size=5, color="black", symbol="diamond"),
        name="Original Cameras (Centers)",
    )
)


# Vectorized line segment creation for Plotly
def create_lines(starts, dirs, scale, color, name, dash="solid"):
    ends = starts + dirs * scale
    x_lines = np.empty(len(starts) * 3)
    x_lines[0::3] = starts[:, 0]
    x_lines[1::3] = ends[:, 0]
    x_lines[2::3] = np.nan
    y_lines = np.empty(len(starts) * 3)
    y_lines[0::3] = starts[:, 1]
    y_lines[1::3] = ends[:, 1]
    y_lines[2::3] = np.nan
    z_lines = np.empty(len(starts) * 3)
    z_lines[0::3] = starts[:, 2]
    z_lines[1::3] = ends[:, 2]
    z_lines[2::3] = np.nan

    return go.Scatter3d(
        x=x_lines,
        y=y_lines,
        z=z_lines,
        mode="lines",
        line=dict(color=color, width=3, dash=dash),
        name=name,
    )


scale = 0.05
# Original Cameras (RGB axes)
fig.add_trace(create_lines(translations, x_dirs, scale, "red", "Orig X-axis"))
fig.add_trace(create_lines(translations, y_dirs, scale, "green", "Orig Y-axis"))
fig.add_trace(create_lines(translations, z_dirs, scale, "blue", "Orig Z-axis"))

# Add NPZ Camera Poses in RED
glb_translations = glb_camera_poses[:, :3, 3]
glb_rotations = glb_camera_poses[:, :3, :3]

glb_x_dirs = glb_rotations[:, :, 0]
glb_y_dirs = glb_rotations[:, :, 1]
glb_z_dirs = glb_rotations[:, :, 2]

# NPZ Camera Centers
fig.add_trace(
    go.Scatter3d(
        x=glb_translations[:, 0],
        y=glb_translations[:, 1],
        z=glb_translations[:, 2],
        mode="markers",
        marker=dict(size=6, color="red", symbol="circle"),
        name="DA3 Predicted Cameras",
    )
)

# NPZ Camera Axes (all in red/variants to distinguish)
fig.add_trace(
    create_lines(glb_translations, glb_x_dirs, scale, "red", "DA3 X-axis", dash="solid")
)
fig.add_trace(
    create_lines(
        glb_translations, glb_y_dirs, scale, "darkred", "DA3 Y-axis", dash="dash"
    )
)
fig.add_trace(
    create_lines(
        glb_translations, glb_z_dirs, scale, "salmon", "DA3 Z-axis", dash="dot"
    )
)

# Enforce equal aspect ratio
fig.update_layout(
    title="3D Point Cloud (NPZ) with Original & DA3 Cameras",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
    width=1000,
    height=800,
    showlegend=True,
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()

BadZipFile: Bad CRC-32 for file 'depth.npy'

In [ ]:
import torch
import open3d as o3d

def estimate_normals_pca(points: torch.Tensor, k: int = 30) -> torch.Tensor:
    """
    Estimates unoriented normals for a batched point cloud using PCA.
    
    Args:
        points: Float32 tensor of coordinates. Shape: [B, N, 3]
        k: Number of nearest neighbors.
        
    Returns:
        normals: Float32 tensor of unoriented normal vectors. Shape: [B, N, 3]
    """
    assert points.dim() == 3, "Input must have shape [B, N, 3]"
    assert points.dtype == torch.float32, "Input must be float32 precision"
    
    device = points.device
    B, N, _ = points.shape # [B, N, 3]
    
    # Vectorized pairwise distance matrix: [B, N, N]
    # Note: For N > 50,000, this fully vectorized O(N^2) operation requires significant VRAM
    dists = torch.cdist(points, points)
    
    # k-nearest neighbors: [B, N, k]
    _, idx = torch.topk(-dists, k=k, dim=-1)
    
    # Gather neighbor coordinates: [B, N, k, 3]
    batch_indices = torch.arange(B, device=device).view(B, 1, 1).expand(B, N, k)
    neighbors = points[batch_indices, idx] 
    
    # Local covariance matrices: [B, N, 3, 3]
    centroid = neighbors.mean(dim=2, keepdim=True) # [B, N, 1, 3]
    centered = neighbors - centroid # [B, N, k, 3]
    
    # Matmul: [B, N, 3, k] @ [B, N, k, 3] -> [B, N, 3, 3]
    cov = torch.matmul(centered.transpose(-2, -1), centered) / (k - 1)
    
    # Eigen decomposition (eigenvalues in ascending order): [B, N, 3, 3]
    _, eigenvectors = torch.linalg.eigh(cov) 
    
    # Normal is the eigenvector of the smallest eigenvalue: [B, N, 3]
    normals = eigenvectors[..., 0] 
    
    # Normalize to unit length: [B, N, 3]
    normals = torch.nn.functional.normalize(normals, p=2, dim=-1)
    
    return normals

def load_glb_to_batched_tensor(file_path: str, num_points: int = 10000) -> torch.Tensor:
    """
    Reads a .glb mesh geometry, uniformly samples a point cloud, 
    and returns a GPU-optimized PyTorch tensor with a batch dimension.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Load the file as a triangle mesh instead of a point cloud
    mesh = o3d.io.read_triangle_mesh(file_path)
    assert not mesh.is_empty(), f"Failed to load or empty geometry detected in: {file_path}"

    # 2. Sample points from the mesh surfaces
    pcd = mesh.sample_points_uniformly(number_of_points=num_points)

    # 3. Transfer directly to PyTorch via the array interface (bypassing explicit numpy import)
    # Shape: (N, 3)
    points_tensor = torch.asarray(pcd.points, dtype=torch.float32, device=device)

    # 4. Enforce batch dimension first requirement
    # Shape: (1, N, 3)
    batched_tensor = points_tensor.unsqueeze(0)

    return batched_tensor

def glb_to_obj_reconstruction(
    input_glb_path: str,
    output_obj_path: str,
    k_neighbors: int = 30,
    poisson_depth: int = 9
) -> None:
    """
    Reads a GLB point cloud, computes normals via PyTorch PCA, orients them,
    reconstructs a Poisson surface mesh, and exports an OBJ file.
    
    Args:
        input_glb_path: Filepath to the input GLB point cloud.
        output_obj_path: Filepath for the output OBJ mesh.
        k_neighbors: Number of neighbors for PCA and orientation.
        poisson_depth: Depth of the Poisson octree.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Load point cloud using Open3D
    assert not pcd.is_empty(), "Failed to load point cloud or file is empty."
    
    # Extract points to a native Python list to bypass NumPy, then to PyTorch Tensor
    points_list = list(pcd.points)
    
    # Add batch dimension to satisfy [B, N, 3] requirement
    points_tensor = torch.tensor(
        points_list, 
        dtype=torch.float32, 
        device=device
    ).unsqueeze(0) # [1, N, 3]
    
    # 2. Compute normals entirely in PyTorch
    normals_tensor = estimate_normals_pca(points_tensor, k=k_neighbors) # [1, N, 3]
    
    # 3. Transfer back to CPU and remove batch dimension
    normals_single = normals_tensor[0].detach().cpu().tolist() # [N, 3]
    pcd.normals = o3d.utility.Vector3dVector(normals_single)
    
    # 4. Orient normals consistently (Requires sequential MST traversal)
    pcd.orient_normals_consistent_tangent_plane(k=k_neighbors)
    
    # 5. Poisson Surface Reconstruction
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd, 
        depth=poisson_depth
    )
    
    # 6. Cleanup low-density artifacts using PyTorch
    # Convert DoubleVector to list, then to Tensor to avoid NumPy requirement
    densities_tensor = torch.tensor(list(densities), dtype=torch.float32) # [V]
    quantile_val = torch.quantile(densities_tensor, 0.01) # Scalar
    
    vertices_to_remove = (densities_tensor < quantile_val).tolist() # [V]
    mesh.remove_vertices_by_mask(vertices_to_remove)
    
    # 7. Export the final manifold mesh
    o3d.io.write_triangle_mesh(output_obj_path, mesh)

# Execution
glb_to_obj_reconstruction("/home/arota/spectra/output/vggt.glb", "output.obj")

[Open3D WARNING] Read geometry::PointCloud failed: unknown file extension for /home/arota/spectra/output/vggt.glb (format: auto).


AssertionError: Failed to load point cloud or file is empty.

In [53]:
pcd = o3d.io.read_point_cloud("/home/arota/spectra/output/vggt")

[Open3D WARNING] Read geometry::PointCloud failed: unknown file extension for /home/arota/spectra/output/vggt (format: auto).


In [ ]:
import open3d as o3d

# 1. Load the point cloud from GLB
# Note: GLB often contains a mesh; if it's points, Open3D reads them directly
pcd = o3d.io.read_point_cloud("/home/arota/spectra/output/vggt.glb")

# 2. Estimate Normals (Required for Poisson)
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30)
)

# 3. Surface Reconstruction (Poisson)
# depth: higher values = higher resolution/detail
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)

# 4. Cleanup (Remove low-density artifacts)
vertices_to_remove = densities < np.quantile(densities, 0.01)
mesh.remove_vertices_by_mask(vertices_to_remove)

# 5. Export
o3d.io.write_triangle_mesh("output.obj", mesh)

[Open3D WARNING] Read geometry::PointCloud failed: unknown file extension for /home/arota/spectra/output/vggt.glb (format: auto).
[Open3D WARNING] [KDTreeFlann::SetRawData] Failed due to no data.


RuntimeError: [1;31m[Open3D Error] (static std::tuple<std::shared_ptr<open3d::geometry::TriangleMesh>, std::vector<double, std::allocator<double> > > open3d::geometry::TriangleMesh::CreateFromPointCloudPoisson(const open3d::geometry::PointCloud&, size_t, float, float, bool, int)) /root/Open3D/cpp/open3d/geometry/SurfaceReconstructionPoisson.cpp:731: Point cloud has no normals
[0;m

In [3]:
import torch
import sys
sys.path.append("/home/arota/spectra/vggt")
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images

device = "cuda" if torch.cuda.is_available() else "cpu"
# bfloat16 is supported on Ampere GPUs (Compute Capability 8.0+) 
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

# Initialize the model and load the pretrained weights.
# This will automatically download the model weights the first time it's run, which may take a while.
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)



In [7]:
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map
from pathlib import Path
import torchvision.io as io
import torch

# Load images from Sample
data_dir = Path("Sample")
image_files = sorted(data_dir.glob("image_*.png"))
pose_files = sorted(data_dir.glob("pose_*.txt"))

import torch.nn.functional as F

images_list = [io.read_image(str(p)).to(device) for p in image_files]
images = torch.stack(images_list)

# Resize to multiples of 14
H, W = images.shape[-2:]
new_H = (H // 14) * 14
new_W = (W // 14) * 14

# Interpolate requires float
images = F.interpolate(images.float(), size=(new_H, new_W), mode='bilinear', align_corners=False).to(images.dtype)

# Load poses from Sample
poses_list = [
    torch.tensor(
        [float(x) for x in p.read_text().strip().split()],
        dtype=torch.float32,
        device=device,
    )
    for p in pose_files
]
poses_tensor = torch.stack(poses_list)

translation_scale = 1.0
poses_tensor4 = xyzeuler_to_hmat(
    poses_tensor, convention="ROLLPITCHYAW", translation_scale=translation_scale
)

cam2ee = (
    torch.tensor(
        [
            [-0.4916, -0.85373, -0.17158, 0.06853 * translation_scale],
            [0.87073, -0.4844, -0.08462, 0.054789 * translation_scale],
            [-0.01088, -0.190998, 0.98153, 0.098354 * translation_scale],
            [0, 0, 0, 1],
        ],
        dtype=torch.float32,
        device=device,
    )
    .unsqueeze(0)
    .repeat(len(poses_tensor), 1, 1)
)

# Extrinsic matrices (world to camera)
poses_tensor_prem = torch.matmul(poses_tensor4, cam2ee)
extrinsic = torch.linalg.inv(poses_tensor_prem).unsqueeze(0)  # [1, N, 4, 4]

# Scale intrinsics according to resize
scale_x = new_W / W
scale_y = new_H / H

# Intrinsic matrices
intrinsic = (
    torch.tensor(
        [
            [389.17394523 * scale_x, 0.0, 315.73276083 * scale_x],
            [0.0, 389.36088727 * scale_y, 246.82430788 * scale_y],
            [0.0, 0.0, 1.0],
        ],
        dtype=torch.float32,
        device=device,
    )
    .unsqueeze(0)
    .repeat(len(poses_tensor), 1, 1)
    .unsqueeze(0)  # [1, N, 3, 3]
)

with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        images = images[None]  # add batch dimension
        aggregated_tokens_list, ps_idx = model.aggregator(images)
                
    # Predict Depth Maps
    depth_map, depth_conf = model.depth_head(aggregated_tokens_list, images, ps_idx)

    # Predict Point Maps
    # point_map, point_conf = model.point_head(aggregated_tokens_list, images, ps_idx)
        
    # Construct 3D Points from Depth Maps and Cameras
    # which usually leads to more accurate 3D points than point map branch
    point_map_by_unprojection = unproject_depth_map_to_point_map(depth_map.squeeze(0), 
                                                                extrinsic.squeeze(0), 
                                                                intrinsic.squeeze(0))

    # Predict Tracks
    # choose your own points to track, with shape (N, 2) for one scene
    # query_points = torch.FloatTensor([[100.0, 200.0], 
    #                                     [60.72, 259.94]]).to(device)
    # track_list, vis_score, conf_score = model.track_head(aggregated_tokens_list, images, ps_idx, query_points=query_points[None])

(20, 476, 630, 3)

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Extract points and colors
# point_map_by_unprojection shape: [N, H, W, 3] or [N, 3, H, W]
pts = point_map_by_unprojection.detach().cpu()
if pts.shape[1] == 3:
    pts = pts.permute(0, 2, 3, 1)  # [N, H, W, 3]

N, H, W, _ = pts.shape
pts = pts.reshape(-1, 3).numpy()

# Extract colors from images
imgs = images.squeeze(0).detach().cpu()  # [N, C, H, W]
if imgs.shape[1] == 3:
    imgs = imgs.permute(0, 2, 3, 1)  # [N, H, W, 3]
colors = imgs.reshape(-1, 3).numpy()
if colors.max() <= 1.0:
    colors = (colors * 255).astype(np.uint8)

# Subsample points to avoid browser crash
skip = 8
mask = np.zeros((N, H, W), dtype=bool)
mask[:, ::skip, ::skip] = True
mask = mask.flatten()

pts_sub = pts[mask]
colors_sub = colors[mask]

# Further subsample if still too large
MAX_POINTS = 100_000
if len(pts_sub) > MAX_POINTS:
    indices = np.random.choice(len(pts_sub), MAX_POINTS, replace=False)
    pts_sub = pts_sub[indices]
    colors_sub = colors_sub[indices]

marker_colors = [f"rgb({c[0]},{c[1]},{c[2]})" for c in colors_sub]

fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=pts_sub[:, 0],
        y=pts_sub[:, 1],
        z=pts_sub[:, 2],
        mode="markers",
        marker=dict(size=2, color=marker_colors, opacity=0.8),
        name="Reconstructed Points",
    )
)

fig.update_layout(
    title="3D Reconstruction from VGGT",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()